In [1]:
import numpy
import sys
import os
import matplotlib.pyplot as plt
# import skimage
import nrrd
from scipy import ndimage
import meshio
from skimage.morphology import skeletonize
from scipy.interpolate import splprep, splev
from sklearn.decomposition import PCA
from scipy.spatial import cKDTree
from scipy.interpolate import UnivariateSpline
import trimesh
from scipy.ndimage import median_filter

from scipy.spatial import cKDTree
from scipy.interpolate import UnivariateSpline
from skimage import measure
import tomllib
from trimesh.path.entities import Line

import pyvista as pv

from scipy.ndimage import uniform_filter1d

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import getSurfaceMesh, export_centerline,  tetra_mesh_from_stl, set_the_offset, tetra_shell_from_two_surfaces
from image_processing_functions import load_vti

from extended_muscle_meshing import (transfer_muscle_profiles, generate_extended_muscle_mesh,
                                     remesh_tube_as_solid, gmsh_mesh_from_surface, mesh_report,
                                     find_eye_intersection_index_mesh, centerline_deviation)

with open("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/config.toml", mode="rb") as f:
    config = tomllib.load(f)

sequence = "T1"

if sequence == "T2":
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file_T2"]
    output_folder   = config["data"]["output_folder_ext"]
    segment_filename = config["data"]["segment_filename_T2"]
    segment_folder  = config["data"]["segment_folder_T2"]
    centerline_folder = config["data"]["centerline_folder"]

else:
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file"]
    output_folder   = config["data"]["output_folder_ext"]
    segment_filename = config["data"]["segment_filename"]
    segment_folder  = config["data"]["segment_folder"]
    centerline_folder = config["data"]["centerline_folder"]

export_name = base + output_folder + segment_filename[:-8]

eyeball_stl = base + output_folder +"re-meshed_adaptive/id_2_shell__h_0-4.stl"

out_folder = base + output_folder + "re-meshed_adaptive" + "/"
os.makedirs(out_folder, exist_ok=True)

## Load image parameters

In [2]:
data, header = nrrd.read(base + segment_folder + segment_filename)

dirs = [numpy.array(d, dtype=float) for d in header["space directions"] if d is not None and not numpy.any(numpy.isnan(numpy.array(d, dtype=float)))]
dirs = numpy.array(dirs)

pixel_spacing = numpy.linalg.norm(dirs, axis=1)           # mm per voxel, in array axis order
direction = (dirs / pixel_spacing[:, None]).T          # columns = unit axis directions
origin = numpy.array(header["space origin"], dtype=float)

print("original pixel spacing = ", pixel_spacing)
print("segmentation origin = ", origin)

original pixel spacing =  [0.3125     0.3125     0.50000006]
segmentation origin =  [-55.95619965 -90.20069885  20.49090004]


In [3]:

n_theta_points = 40

voxel_mm = float(numpy.min(pixel_spacing))   # isotropic spacing after resampling
contact_axis_1 = 8 * voxel_mm
contact_axis_2 = 3 * voxel_mm
print("contact semi-axes (mm):", contact_axis_1, contact_axis_2)

contact semi-axes (mm): 2.499999999999999 0.9374999999999997


## Load segments


In [4]:
segments = []
i = 0
while f"Segment{i}_Name" in header:
    name  = header[f"Segment{i}_Name"]
    label = int(header.get(f"Segment{i}_LabelValue", i + 1))
    layer = int(header.get(f"Segment{i}_Layer", 0))
    vol   = data[layer] if data.ndim == 4 else data
    segments.append((name, vol == label))
    print(f"{name}: layer={layer}, value={label}, voxels={segments[-1][1].sum()}")
    i += 1

combined = numpy.zeros(segments[0][1].shape, dtype=numpy.uint8)
for new_id, (name, mask) in enumerate(segments, start=1):
    combined[mask] = new_id          # later segments overwrite earlier ones where they overlap

data = combined
id_max = int(data.max())


optical nerve: layer=0, value=1, voxels=15624
eyeball: layer=0, value=3, voxels=160561
lateral rectus: layer=0, value=4, voxels=14436
medial rectus: layer=0, value=5, voxels=13164
inferior rectus: layer=0, value=6, voxels=11961
superior rectus: layer=0, value=7, voxels=13568
lens: layer=0, value=2, voxels=1508


## Resample

In [5]:
pixel_spacing = numpy.asarray(pixel_spacing, dtype=float)
target = pixel_spacing.min()

if not numpy.allclose(pixel_spacing, target):
    factors = pixel_spacing / target
    data = ndimage.zoom(data, factors, order=0)
    print("resampling factors = ", factors)
    print("new dimensions = ", data.shape)

pixel_spacing = target

print("Final uniform pixel_spacing = ", pixel_spacing)

resampling factors =  [1.         1.         1.60000019]
new dimensions =  (145, 209, 162)
Final uniform pixel_spacing =  0.3124999999999999


## Define IDs

In [6]:
eyeball_id = 2
lens_id = 7

muscle_ids = [3,4]
ON_id = 1



## Create initial meshes

In [7]:
muslce_masks = []

for id in muscle_ids: 
    mask = numpy.zeros(data.shape, dtype=numpy.uint8)
    mask[(data>id-0.5) & (data<id+0.5 )]=1
    muslce_masks.append(mask)

ON_mask = numpy.zeros(data.shape, dtype=numpy.uint8)
ON_mask[(data>ON_id-0.5) & (data<ON_id+0.5 )]=1

eyeball_mask = numpy.zeros(data.shape, dtype=numpy.uint8)
eyeball_mask[(data>eyeball_id-0.5) & (data<eyeball_id+0.5 )]=1


# Part II

In [8]:
def get_local_frame(direction):
    """Calculates two vectors perpendicular to the spine direction."""
    # Pick a non-parallel vector to start the cross product
    # vec = numpy.array([1, 0, 0]) if abs(direction[0]) < 0.95 else numpy.array([0, 1, 0])

    vec_1 = numpy.array([1, 1, 0])
    vec = vec_1/numpy.linalg.norm(vec_1)

    v1 = numpy.cross(direction, vec)
    v1 /= numpy.linalg.norm(v1)
    v2 = numpy.cross(direction, v1)
    return v1, v2


## Find intersection of the extended centerline and the eyeball


In [9]:
def find_eye_intersection_index(centerline_extended, mask_eye):
    """
    Returns the index along the extended centerline that intersects the eye.
    """
    for idx, point in enumerate(centerline_extended):
        z, y, x = numpy.round(point).astype(int)
        
        # Check bounds
        if 0 <= z < mask_eye.shape[0] and 0 <= y < mask_eye.shape[1] and 0 <= x < mask_eye.shape[2]:
            if mask_eye[z, y, x] == 1:
                return idx  
                
    return len(centerline_extended) - 1


## Elliptic contact surface pt 1

In [10]:
def compute_local_tangent_from_section(section_2d_path, contact_point_mm):
    """
    Computes the normalized tangent vector using the topological 
    line entities of the trimesh section.
    """
    vertices = section_2d_path.vertices
    
    # 1. Find the entity segment closest to the contact point
    min_dist = float('inf')
    best_tangent = None
    
    for entity in section_2d_path.entities:
        # Entities contain ordered point indices, e.g. [idx_a, idx_b]
        points = entity.points
        for i in range(len(points) - 1):
            pA = vertices[points[i]]
            pB = vertices[points[i + 1]]
            
            # Midpoint of segment
            seg_mid = (pA + pB) / 2.0
            dist = numpy.linalg.norm(seg_mid - contact_point_mm)
            
            if dist < min_dist:
                min_dist = dist
                vec = pB - pA
                norm_vec = numpy.linalg.norm(vec)
                if norm_vec > 1e-8:
                    best_tangent = vec / norm_vec

    if best_tangent is None:
        raise ValueError("Could not extract a valid tangent from the section curve.")

    return best_tangent

## Elliptic contact surface pt 2

In [11]:
def get_eyeball_elliptic_contact_surface(
    eyeball_stl_path, 
    centerline_extended, 
    eye_intersection_idx, 
    pixel_spacing, 
    curve_tangent,             # Direction along section curve B
    semi_major_mm=3.0,         # Ellipse length along tangent
    semi_minor_mm=1.5,         # Ellipse width perpendicular to tangent
    n_theta=50, 
    offset_mm=1.0,             # <--- NEW: Outward gap distance from eyeball surface
    output_vtk_path="eyeball_contact.vtk"
):
    """
    Extracts an elliptic contact patch aligned with a local path tangent direction,
    and offsets it outward from the eyeball surface to create a clearance gap.
    """
    # Load the eyeball mesh
    mesh_trimesh = trimesh.load(eyeball_stl_path)
    vertices = mesh_trimesh.vertices
    faces = mesh_trimesh.faces
    
    # 1. Contact point setup
    contact_voxel_coord = centerline_extended[eye_intersection_idx]
    contact_point_mm = contact_voxel_coord * pixel_spacing
    
    mesh_tree = cKDTree(vertices)
    dist, closest_vertex_idx = mesh_tree.query(contact_point_mm)
    mesh_contact_point = vertices[closest_vertex_idx]
    
    # -------------------------------------------------------------------------
    # 1. Create Local Coordinate System (u, v, n) aligned with Section Tangent
    # -------------------------------------------------------------------------
    u_axis = curve_tangent / numpy.linalg.norm(curve_tangent)
    
    # Compute surface normal at contact point using vertex normals
    surface_normal = mesh_trimesh.vertex_normals[closest_vertex_idx]
    surface_normal = surface_normal / numpy.linalg.norm(surface_normal)
    
    # v: Minor axis perpendicular to tangent and surface normal
    v_axis = numpy.cross(surface_normal, u_axis)
    v_axis /= numpy.linalg.norm(v_axis)
    
    # Re-orthonormalize u_axis to guarantee pure perpendicular axes
    u_axis = numpy.cross(v_axis, surface_normal)
    u_axis /= numpy.linalg.norm(u_axis)

    # -------------------------------------------------------------------------
    # 2. Extract Elliptic Disc Sub-Mesh Patch
    # -------------------------------------------------------------------------
    face_centers = mesh_trimesh.triangles_center
    
    face_tree = cKDTree(face_centers)
    candidate_indices = face_tree.query_ball_point(mesh_contact_point, r=max(semi_major_mm, semi_minor_mm) * 1.2)
    
    patch_face_indices = []
    for f_idx in candidate_indices:
        rel_pos = face_centers[f_idx] - mesh_contact_point
        
        dist_u = numpy.dot(rel_pos, u_axis)
        dist_v = numpy.dot(rel_pos, v_axis)
        
        ellipse_metric = (dist_u / semi_major_mm)**2 + (dist_v / semi_minor_mm)**2
        if ellipse_metric <= 1.0:
            patch_face_indices.append(f_idx)
            
    if len(patch_face_indices) < 3:
        raise ValueError("Ellipse semi-axes are too small; no mesh faces captured.")
        
    patch_mesh = mesh_trimesh.submesh([patch_face_indices], append=True)

    # -------------------------------------------------------------------------
    # 3. Boundary Extraction (Unique Edges)
    # -------------------------------------------------------------------------
    edges_sorted = patch_mesh.edges_sorted
    unique_edge_indices = trimesh.grouping.group_rows(edges_sorted, require_count=1)
    boundary_edges = patch_mesh.edges[unique_edge_indices]

    # -------------------------------------------------------------------------
    # 4. Adjacency Loop Walker
    # -------------------------------------------------------------------------
    adjacency = {}
    for u, v in boundary_edges:
        adjacency.setdefault(u, []).append(v)
        adjacency.setdefault(v, []).append(u)
        
    start_node = boundary_edges[0][0]
    ordered_patch_vertex_indices = [start_node]
    
    current_node = adjacency[start_node][0]
    prev_node = start_node
    
    while current_node != start_node:
        ordered_patch_vertex_indices.append(current_node)
        neighbors = adjacency[current_node]
        next_node = neighbors[0] if neighbors[0] != prev_node else neighbors[1]
        prev_node = current_node
        current_node = next_node
        
    raw_rim_vertices = patch_mesh.vertices[ordered_patch_vertex_indices]

    # -------------------------------------------------------------------------
    # 5. Angular Resampling aligned with Local Ellipse Frame
    # -------------------------------------------------------------------------
    rel_rim = raw_rim_vertices - mesh_contact_point
    rim_angles = numpy.arctan2(rel_rim @ v_axis, rel_rim @ u_axis)
    
    sort_idx = numpy.argsort(rim_angles)
    sorted_rim_angles = rim_angles[sort_idx]
    sorted_rim_vertices = raw_rim_vertices[sort_idx]
    
    target_angles = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    ordered_contour_vertices = numpy.zeros((n_theta, 3))
    
    for i in range(3):
        ordered_contour_vertices[:, i] = numpy.interp(
            target_angles, sorted_rim_angles, sorted_rim_vertices[:, i], period=2 * numpy.pi
        )

    # -------------------------------------------------------------------------
    # 6. APPLY OUTWARD OFFSET (FOR MUSCLE ATTACHMENT)
    # -------------------------------------------------------------------------
    if offset_mm > 0:
        # Option A: Smooth directional shift using local vertex normals (preserves local surface curvature)
        # Interpolate vertex normals for the resampled rim points
        rim_tree = cKDTree(patch_mesh.vertices)
        _, rim_to_patch_indices = rim_tree.query(ordered_contour_vertices)
        contour_normals = patch_mesh.vertex_normals[rim_to_patch_indices]
        
        # Normalize vectors just in case
        contour_normals /= numpy.linalg.norm(contour_normals, axis=1, keepdims=True)
        
        # Push contour points outward along their local surface normals
        muscle_contour_vertices = ordered_contour_vertices + contour_normals * offset_mm
        
        # Push central point outward along main surface normal
        muscle_contact_point = mesh_contact_point + surface_normal * offset_mm
    else:
        muscle_contour_vertices = ordered_contour_vertices
        muscle_contact_point = mesh_contact_point

    # -------------------------------------------------------------------------
    # 7. Export Attributed VTK Data
    # -------------------------------------------------------------------------
    nv = faces.shape[0]
    padding = numpy.ones((nv, 1), dtype=int) * 3
    pv_faces = numpy.hstack((padding, faces)).ravel()
    grid = pv.PolyData(vertices, pv_faces)
    
    global_rim_indices = []
    global_tree = cKDTree(vertices)
    for pt in ordered_contour_vertices:
        _, glob_idx = global_tree.query(pt)
        global_rim_indices.append(glob_idx)
        
    global_patch_v_indices = numpy.unique(faces[patch_face_indices].ravel())
    
    node_registration = numpy.zeros(grid.n_points, dtype=numpy.int32)
    node_registration[global_patch_v_indices] = 2  # Elliptic patch interior
    node_registration[global_rim_indices] = 3      # Boundary rim
    node_registration[closest_vertex_idx] = 1     # Contact center point
    
    grid.point_data["NodeRegistration"] = node_registration
    grid.save(output_vtk_path)
    
    print(f"--> [SUCCESS] Saved Elliptic VTK mesh to: {output_vtk_path}")
    
    # Return the OFFSETTED points so the muscle mesh uses them
    return muscle_contact_point, muscle_contour_vertices

## Contact surface on the eyeball mesh

In [12]:
def get_eyeball_elliptic_contact_surface(
    eyeball_stl_path, 
    centerline_extended, 
    eye_intersection_idx, 
    pixel_spacing, 
    curve_tangent,             # Direction along section curve B
    semi_major_mm=2.0,         # Length along section curve
    semi_minor_mm=2.0,         # Width perpendicular to section curve
    n_theta=50, 
    output_vtk_path="eyeball_contact.vtk"
):
    # Load mesh
    mesh_trimesh = trimesh.load(eyeball_stl_path)
    vertices = mesh_trimesh.vertices
    faces = mesh_trimesh.faces
    
    # 1. Contact point setup
    contact_voxel_coord = centerline_extended[eye_intersection_idx]
    contact_point_mm = contact_voxel_coord * pixel_spacing
    
    mesh_tree = cKDTree(vertices)
    dist, closest_vertex_idx = mesh_tree.query(contact_point_mm)
    mesh_contact_point = vertices[closest_vertex_idx]
    
    # -------------------------------------------------------------------------
    # 1. Local Tangent Frame for Elliptic Spatial Filter
    # -------------------------------------------------------------------------
    u_axis = curve_tangent / numpy.linalg.norm(curve_tangent)
    surface_normal = mesh_trimesh.vertex_normals[closest_vertex_idx]
    surface_normal = surface_normal/numpy.linalg.norm(surface_normal)
    
    v_axis = numpy.cross(surface_normal, u_axis)
    v_axis /= numpy.linalg.norm(v_axis)
    u_axis = numpy.cross(v_axis, surface_normal)
    u_axis /= numpy.linalg.norm(u_axis)

    # -------------------------------------------------------------------------
    # 2. Extract Elliptic Submesh Patch
    # -------------------------------------------------------------------------
    face_centers = mesh_trimesh.triangles_center
    face_tree = cKDTree(face_centers)
    candidate_indices = face_tree.query_ball_point(mesh_contact_point, r=max(semi_major_mm, semi_minor_mm) * 1.2)
    
    patch_face_indices = []
    for f_idx in candidate_indices:
        rel_pos = face_centers[f_idx] - mesh_contact_point
        dist_u = numpy.dot(rel_pos, u_axis)
        dist_v = numpy.dot(rel_pos, v_axis)
        
        if (dist_u / semi_major_mm)**2 + (dist_v / semi_minor_mm)**2 <= 1.0:
            patch_face_indices.append(f_idx)
            
    if len(patch_face_indices) < 3:
        raise ValueError("Ellipse semi-axes are too small; no mesh faces captured.")
        
    patch_mesh = mesh_trimesh.submesh([patch_face_indices], append=True)

    # -------------------------------------------------------------------------
    # 3. Boundary Extraction & Adjacency Loop Walk
    # -------------------------------------------------------------------------
    edges_sorted = patch_mesh.edges_sorted
    unique_edge_indices = trimesh.grouping.group_rows(edges_sorted, require_count=1)
    boundary_edges = patch_mesh.edges[unique_edge_indices]

    adjacency = {}
    for u, v in boundary_edges:
        adjacency.setdefault(u, []).append(v)
        adjacency.setdefault(v, []).append(u)
        
    start_node = boundary_edges[0][0]
    ordered_patch_vertex_indices = [start_node]
    
    current_node = adjacency[start_node][0]
    prev_node = start_node
    
    while current_node != start_node:
        ordered_patch_vertex_indices.append(current_node)
        neighbors = adjacency[current_node]
        next_node = neighbors[0] if neighbors[0] != prev_node else neighbors[1]
        prev_node = current_node
        current_node = next_node
        
    raw_rim_vertices = patch_mesh.vertices[ordered_patch_vertex_indices]

    # -------------------------------------------------------------------------
    # 4. Angular Synchronization (EXACT MATCH TO YOUR ORIGINAL CODE)
    # -------------------------------------------------------------------------
    # We use the centerline tangent (same frame reference as original code)
    idx_next = min(eye_intersection_idx + 1, len(centerline_extended) - 1)
    idx_prev = max(eye_intersection_idx - 1, 0)
    centerline_tangent = (centerline_extended[idx_next] - centerline_extended[idx_prev]) * pixel_spacing
    centerline_tangent /= numpy.linalg.norm(centerline_tangent)
    
    if numpy.abs(centerline_tangent[0]) < 0.9:
        v1 = numpy.cross(centerline_tangent, [1, 0, 0])
    else:
        v1 = numpy.cross(centerline_tangent, [0, 1, 0])
    v1 /= numpy.linalg.norm(v1)
    v2 = numpy.cross(centerline_tangent, v1)
    v2 /= numpy.linalg.norm(v2)
    
    # Project rim vertices into original (v1, v2) coordinate frame
    rel_rim = raw_rim_vertices - mesh_contact_point
    rim_angles = numpy.arctan2(rel_rim @ v2, rel_rim @ v1)
    
    sort_idx = numpy.argsort(rim_angles)
    sorted_rim_angles = rim_angles[sort_idx]
    sorted_rim_vertices = raw_rim_vertices[sort_idx]
    
    target_angles = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    ordered_contour_vertices = numpy.zeros((n_theta, 3))
    
    for i in range(3):
        ordered_contour_vertices[:, i] = numpy.interp(
            target_angles, sorted_rim_angles, sorted_rim_vertices[:, i], period=2 * numpy.pi
        )

    # -------------------------------------------------------------------------
    # 5. Export Attributed VTK Data
    # -------------------------------------------------------------------------
    nv = faces.shape[0]
    padding = numpy.ones((nv, 1), dtype=int) * 3
    pv_faces = numpy.hstack((padding, faces)).ravel()
    grid = pv.PolyData(vertices, pv_faces)
    
    global_rim_indices = []
    global_tree = cKDTree(vertices)
    for pt in ordered_contour_vertices:
        _, glob_idx = global_tree.query(pt)
        global_rim_indices.append(glob_idx)
        
    global_patch_v_indices = numpy.unique(faces[patch_face_indices].ravel())
    
    node_registration = numpy.zeros(grid.n_points, dtype=numpy.int32)
    node_registration[global_patch_v_indices] = 2
    node_registration[global_rim_indices] = 3
    node_registration[closest_vertex_idx] = 1
    
    grid.point_data["NodeRegistration"] = node_registration
    grid.save(output_vtk_path)
    
    return mesh_contact_point, ordered_contour_vertices




## Set meshing parameters

In [13]:
n_theta_points = 40
smoothing_factor = 15
spline_order = 3

## Get contact points & surfaces

In [14]:
def load_section(path):
    pts = numpy.load(path)                                  # (n, 3)
    return trimesh.path.Path3D(entities=[Line(numpy.arange(len(pts)))], vertices=pts)


## Meshing loop

In [ ]:
# ===========================================================================
# Settings
# ===========================================================================
h = 0.9
h_eyeball = 0.4            # element size at the contact, same as the eyeball mesh

offset = None
n_theta_points = 40

voxel_mm = float(numpy.min(pixel_spacing))
contact_axis_1 = 8 * voxel_mm
contact_axis_2 = 3 * voxel_mm

eyeball_stl = f"{export_name}_surf_id_{eyeball_id}_outer.stl"
eye_mesh = trimesh.load(eyeball_stl)
section = load_section(base + centerline_folder + "section_curve_A.npy")

# out_folder = base + output_folder + "re-meshed_h_" + str(h).replace(".", "-") + "/"
# os.makedirs(out_folder, exist_ok=True)

# ===========================================================================
# Loop over muscles
# ===========================================================================
for idx, muscle_id in enumerate(muscle_ids):
    print(f"--- muscle {muscle_id} ---")

    # 1. Centerlines: original (shape) and extended (position), the latter clipped at the eye
    original = numpy.load(base + centerline_folder + f"centerline_init_id_{muscle_id}.npy")
    extended = numpy.load(base + centerline_folder + f"centerline_exttended_2_id_{muscle_id}.npy")
    eye_idx = find_eye_intersection_index_mesh(extended, eye_mesh, pixel_spacing, min_depth_mm=0.3)
    trimmed = extended[:eye_idx + 1]

    # 2. Measure along the ORIGINAL centerline, filter, map onto the EXTENDED centerline
    centerline, v1, v2, body_radii, info = transfer_muscle_profiles(
        original, muslce_masks[idx], trimmed, pixel_spacing, n_theta=n_theta_points,
        start_trim=6, end_trim=6,          # drop the rounded tips of the segmentation
        match="start",                     # the start should macth on both versions
        start_extension_mm=2.0,
        fit_window=15, min_frac=0.5, max_frac=1.0, extension_end_scale=1.0)

    # 3. Contact ring (ellipse) on the eyeball
    contact_pt_mm = extended[eye_idx] * pixel_spacing
    curve_tangent = compute_local_tangent_from_section(section, contact_pt_mm)
    contact_center, eye_ring = get_eyeball_elliptic_contact_surface(
        eyeball_stl_path=eyeball_stl,
        centerline_extended=extended,
        eye_intersection_idx=eye_idx,
        pixel_spacing=pixel_spacing,
        curve_tangent=curve_tangent,
        semi_major_mm=contact_axis_1,
        semi_minor_mm=contact_axis_2,
        n_theta=n_theta_points,
        output_vtk_path=f"{export_name}_muscle_{muscle_id}_eye_contact.vtk",
    )

    # 4. Ring tube along the extended centerline (tail extrapolated, blended into the eye ring)
    tube, _ = generate_extended_muscle_mesh(
        centerline, v1, v2, body_radii, eye_ring, contact_center, pixel_spacing,
        taper_shape="cone", blend_length_mm=15.0, follow_centerline=True)
    dmax, dmean = centerline_deviation(tube, n_theta_points, trimmed, pixel_spacing)
    print(f"   tube vs extended centerline: max {dmax:.2f} mm, mean {dmean:.2f} mm")

    # 5. Muscle surface, cut by the eyeball mesh; muscle attached to the eye 
    surface = remesh_tube_as_solid(
        tube, n_theta_points, pixel_spacing,
        resolution_mm=0.15,
        eye_surface=eye_mesh,
        eye_clearance_mm=0.1,
        contact_fill_mm=0.3,       # fill gaps to the eye up to ~0.6 mm 
        contact_open_mm=0.3,       # remove remaining thin spikes at the contact
        target_faces=20000,
        close_iterations=2)
    
    # 6. Simulation mesh: h_eyeball at the eye, growing to h
    output_filename = out_folder + "remeshed_h_" + str(h).replace(".", "-") + f"_id_{muscle_id}"
    surf, quality = gmsh_mesh_from_surface(
        surface, output_filename, element_size=h,
        refine_near=eye_mesh,      # distance is measured to the eyeball surface
        h_near=h_eyeball,
        near_distance_mm=0.3,      # keep h_eyeball within 0.3 mm of the eye
        transition_mm=3.0)         # then grow to h over 3 mm
    print("output_filename = ", output_filename)

--- muscle 3 ---
   measured along original centerline: 31.9 mm; mapped onto extended centerline from 3.3 mm (scale 1.000)
   nodes: 155 (head 12, body 90, transition 53), profile rotation -6 to -4 deg
   tube vs extended centerline: max 0.33 mm, mean 0.15 mm
   contact fill: 2.49 mm^3 added between muscle and eye
   gmsh: remeshed surface, angle 60.0
   edge length: mean 0.922 mm (target 0.9), 5-95 %: 0.571 - 1.305 mm  [gmsh size used: 0.756]
   triangles: 3796, gamma min 0.031, mean 0.932, < 0.1: 4
   tets: 9382, gamma min 0.036, mean 0.798, < 0.1: 9
--- muscle 4 ---
   measured along original centerline: 28.1 mm; mapped onto extended centerline from 2.5 mm (scale 1.000)
   nodes: 128 (head 12, body 78, transition 38), profile rotation -1 to 0 deg
   tube vs extended centerline: max 0.23 mm, mean 0.10 mm
   contact fill: 2.37 mm^3 added between muscle and eye
   gmsh: remeshed surface, angle 60.0
   edge length: mean 0.920 mm (target 0.9), 5-95 %: 0.569 - 1.299 mm  [gmsh size used: 0